In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd

base_path = "/Volumes/data_analysis_retail/source_data/raw/Gap_Analysis/"

outlet_df = pd.read_excel(base_path + "Retail_Assessment_Demo_Data.xlsx", sheet_name="Outlet Master", skiprows=1)
sales_df  = pd.read_excel(base_path + "Retail_Assessment_Demo_Data.xlsx", sheet_name="Daily Sales",   skiprows=1)
gap_df    = pd.read_excel(base_path + "Retail_Assessment_Demo_Data.xlsx", sheet_name="Gap Analysis",  skiprows=1)

# ── Clean gap_df ──────────────────────────────────────────────────
# Drop rows where Outlet_ID is null or doesn't start with "OT"
gap_df = gap_df[
    gap_df["Outlet_ID"].notna() &
    gap_df["Outlet_ID"].astype(str).str.startswith("OT")
]

# Same safety filter for outlet and sales just in case
outlet_df = outlet_df[
    outlet_df["Outlet_ID"].notna() &
    outlet_df["Outlet_ID"].astype(str).str.startswith("OT")
]

sales_df = sales_df[
    sales_df["Outlet_ID"].notna() &
    sales_df["Outlet_ID"].astype(str).str.startswith("OT")
]

# Register as Spark temp views
spark.createDataFrame(outlet_df).createOrReplaceTempView("v_outlet_master")
spark.createDataFrame(sales_df).createOrReplaceTempView("v_daily_sales")
spark.createDataFrame(gap_df).createOrReplaceTempView("v_gap_analysis")

# Verify counts
print(f"✅ Outlets : {len(outlet_df)} rows")
print(f"✅ Sales   : {len(sales_df)} rows")
print(f"✅ Gap     : {len(gap_df)} rows  ← should be exactly 50")

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.outlet_gap_summary AS
SELECT
  g.Outlet_ID,
  g.Location                                                              AS region,
  g.Store_Type                                                            AS store_type,
  g.Target_Sales                                                          AS target_sales,
  g.Actual_Sales                                                          AS actual_sales,
  g.Sales_Gap                                                             AS sales_gap,
  ROUND(
    ((g.Actual_Sales - g.Target_Sales) / g.Target_Sales) * 100, 2
  )                                                                       AS gap_pct,
  CASE
    WHEN ((g.Actual_Sales - g.Target_Sales) / g.Target_Sales) * 100 < -20 THEN 'Critical'
    WHEN ((g.Actual_Sales - g.Target_Sales) / g.Target_Sales) * 100 < -10 THEN 'At Risk'
    WHEN ((g.Actual_Sales - g.Target_Sales) / g.Target_Sales) * 100 < 0   THEN 'Slightly Under'
    ELSE                                                                        'On/Above Target'
  END                                                                     AS performance_tier
FROM v_gap_analysis g;

In [0]:
%sql
SELECT performance_tier, COUNT(*) AS outlet_count
FROM data_analysis_retail.gold.outlet_gap_summary
GROUP BY performance_tier
ORDER BY performance_tier;

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.region_performance AS
SELECT
  g.Location                                          AS region,
  COUNT(g.Outlet_ID)                                  AS total_outlets,
  SUM(g.Target_Sales)                                 AS total_target,
  SUM(g.Actual_Sales)                                 AS total_actual,
  SUM(g.Sales_Gap)                                    AS total_gap,
  ROUND(
    (SUM(g.Actual_Sales) - SUM(g.Target_Sales))
    / SUM(g.Target_Sales) * 100, 2
  )                                                   AS region_gap_pct,
  COUNT(CASE WHEN g.`Gap_%` < 0 THEN 1 END)          AS outlets_missing_target
FROM v_gap_analysis g
GROUP BY g.Location;

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.daily_sales_trend AS
SELECT
  s.Date,
  s.Outlet_ID,
  o.Location      AS region,
  o.Store_Type    AS store_type,
  s.Sales_Amount  AS daily_sales,
  s.Transactions  AS transactions,
  s.Customers     AS customers
FROM v_daily_sales s
JOIN v_outlet_master o ON s.Outlet_ID = o.Outlet_ID;

### Outlet master gold layer

In [0]:
import pandas as pd

base_path = "/Volumes/data_analysis_retail/source_data/raw/Outlet_master/"

outlet_df = pd.read_excel(base_path + "Retail_Assessment_Demo_Data.xlsx", sheet_name="Outlet Master", skiprows=1)
sales_df  = pd.read_excel(base_path + "Retail_Assessment_Demo_Data.xlsx", sheet_name="Daily Sales",   skiprows=1)

# Filter junk rows — only real outlet IDs
outlet_df = outlet_df[outlet_df["Outlet_ID"].notna() & outlet_df["Outlet_ID"].astype(str).str.startswith("OT")]
sales_df  = sales_df[sales_df["Outlet_ID"].notna()  & sales_df["Outlet_ID"].astype(str).str.startswith("OT")]

spark.createDataFrame(outlet_df).createOrReplaceTempView("v_outlet_master")
spark.createDataFrame(sales_df).createOrReplaceTempView("v_daily_sales")

print(f"✅ Outlets : {len(outlet_df)} rows — should be 50")
print(f"✅ Sales   : {len(sales_df)} rows — should be 18,250")

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.outlet_top_performers AS
SELECT
  o.Outlet_ID,
  o.Location,
  o.Store_Type,
  o.Target_Sales,
  ROUND(SUM(s.Sales_Amount), 2)                                        AS actual_sales,
  ROUND(SUM(s.Sales_Amount) - o.Target_Sales, 2)                      AS sales_gap,
  ROUND((SUM(s.Sales_Amount) - o.Target_Sales) / o.Target_Sales * 100, 2) AS gap_pct,
  RANK() OVER (ORDER BY SUM(s.Sales_Amount) DESC)                     AS sales_rank,
  CASE
    WHEN (SUM(s.Sales_Amount) - o.Target_Sales) / o.Target_Sales * 100 >= 20 THEN 'Star'
    WHEN (SUM(s.Sales_Amount) - o.Target_Sales) / o.Target_Sales * 100 >= 0  THEN 'On Target'
    WHEN (SUM(s.Sales_Amount) - o.Target_Sales) / o.Target_Sales * 100 >= -20 THEN 'Underperforming'
    ELSE 'Critical'
  END AS performance_label
FROM v_outlet_master o
JOIN v_daily_sales s ON o.Outlet_ID = s.Outlet_ID
GROUP BY o.Outlet_ID, o.Location, o.Store_Type, o.Target_Sales;

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.outlet_monthly_trend AS
SELECT
  s.Outlet_ID,
  o.Location,
  o.Store_Type,
  DATE_FORMAT(s.Date, 'yyyy-MM')          AS sales_month,
  MONTH(s.Date)                           AS month_num,
  ROUND(SUM(s.Sales_Amount), 2)           AS monthly_sales,
  COUNT(s.Transactions)                   AS total_transactions,
  SUM(s.Customers)                        AS total_customers
FROM v_daily_sales s
JOIN v_outlet_master o ON s.Outlet_ID = o.Outlet_ID
GROUP BY s.Outlet_ID, o.Location, o.Store_Type, DATE_FORMAT(s.Date, 'yyyy-MM'), MONTH(s.Date);

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.outlet_age_performance AS
SELECT
  o.Outlet_ID,
  o.Location,
  o.Store_Type,
  o.Opening_Date,
  DATEDIFF(TO_DATE('2024-12-31'), TO_DATE(o.Opening_Date)) / 365.0   AS outlet_age_years,
  ROUND(
    DATEDIFF(TO_DATE('2024-12-31'), TO_DATE(o.Opening_Date)) / 365.0
  )                                                                    AS age_years_rounded,
  CASE
    WHEN DATEDIFF(TO_DATE('2024-12-31'), TO_DATE(o.Opening_Date)) / 365.0 < 2  THEN 'New (< 2 yrs)'
    WHEN DATEDIFF(TO_DATE('2024-12-31'), TO_DATE(o.Opening_Date)) / 365.0 < 5  THEN 'Growing (2–5 yrs)'
    ELSE                                                                              'Mature (5+ yrs)'
  END                                                                  AS age_category,
  o.Target_Sales,
  ROUND(SUM(s.Sales_Amount), 2)                                        AS actual_sales,
  ROUND((SUM(s.Sales_Amount) - o.Target_Sales) / o.Target_Sales * 100, 2) AS gap_pct
FROM v_outlet_master o
JOIN v_daily_sales s ON o.Outlet_ID = s.Outlet_ID
GROUP BY o.Outlet_ID, o.Location, o.Store_Type, o.Opening_Date, o.Target_Sales;

In [0]:
%sql
SELECT 'top_performers'    AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.outlet_top_performers
UNION ALL
SELECT 'monthly_trend'     AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.outlet_monthly_trend
UNION ALL
SELECT 'age_performance'   AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.outlet_age_performance;

### Employee KPI

In [0]:
import pandas as pd

base_path = "/Volumes/data_analysis_retail/source_data/raw/Employee_KPI/"
file = "Retail_Assessment_Demo_Data (1).xlsx"

emp_df    = pd.read_excel(base_path + file, sheet_name="Employee KPI",  skiprows=1)
outlet_df = pd.read_excel(base_path + file, sheet_name="Outlet Master", skiprows=1)
sales_df  = pd.read_excel(base_path + file, sheet_name="Daily Sales",   skiprows=1)

# Clean all three — keep only real rows
emp_df    = emp_df[emp_df["Employee_ID"].notna() & emp_df["Employee_ID"].astype(str).str.startswith("EMP")]
outlet_df = outlet_df[outlet_df["Outlet_ID"].notna() & outlet_df["Outlet_ID"].astype(str).str.startswith("OT")]
sales_df  = sales_df[sales_df["Outlet_ID"].notna()  & sales_df["Outlet_ID"].astype(str).str.startswith("OT")]

spark.createDataFrame(emp_df).createOrReplaceTempView("v_employee_kpi")
spark.createDataFrame(outlet_df).createOrReplaceTempView("v_outlet_master")
spark.createDataFrame(sales_df).createOrReplaceTempView("v_daily_sales")

print(f"✅ Employees : {len(emp_df)} rows  — should be 347")
print(f"✅ Outlets   : {len(outlet_df)} rows — should be 50")
print(f"✅ Sales     : {len(sales_df)} rows — should be 18,250")

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.kpi_by_outlet AS
SELECT
  e.Outlet_ID,
  o.Location,
  o.Store_Type,
  COUNT(e.Employee_ID)                            AS total_employees,
  ROUND(AVG(e.KPI_Achievement_Pct), 2)           AS avg_kpi_achievement,
  ROUND(AVG(e.KPI_Target), 2)                    AS avg_kpi_target,
  ROUND(AVG(e.KPI_Achievement_Pct) - AVG(e.KPI_Target), 2) AS kpi_gap,
  CASE
    WHEN AVG(e.KPI_Achievement_Pct) >= 100 THEN 'Exceeding'
    WHEN AVG(e.KPI_Achievement_Pct) >= 80  THEN 'On Track'
    WHEN AVG(e.KPI_Achievement_Pct) >= 60  THEN 'At Risk'
    ELSE                                         'Critical'
  END AS kpi_status
FROM v_employee_kpi e
JOIN v_outlet_master o ON e.Outlet_ID = o.Outlet_ID
GROUP BY e.Outlet_ID, o.Location, o.Store_Type;

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.employee_performance AS
SELECT
  e.Employee_ID,
  e.Outlet_ID,
  o.Location,
  e.Position,
  e.KPI_Target,
  e.KPI_Achievement_Pct,
  ROUND(e.KPI_Achievement_Pct - e.KPI_Target, 2) AS kpi_vs_target,
  RANK() OVER (ORDER BY e.KPI_Achievement_Pct DESC) AS rank_top,
  RANK() OVER (ORDER BY e.KPI_Achievement_Pct ASC)  AS rank_bottom,
  CASE
    WHEN e.KPI_Achievement_Pct >= 100 THEN 'Top Performer'
    WHEN e.KPI_Achievement_Pct >= 80  THEN 'Good'
    WHEN e.KPI_Achievement_Pct >= 60  THEN 'Average'
    ELSE                                   'Low Performer'
  END AS performance_label
FROM v_employee_kpi e
JOIN v_outlet_master o ON e.Outlet_ID = o.Outlet_ID;

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.kpi_sales_correlation AS
SELECT
  e.Outlet_ID,
  o.Location,
  o.Store_Type,
  ROUND(AVG(e.KPI_Achievement_Pct), 2)  AS avg_kpi_achievement,
  ROUND(SUM(s.Sales_Amount), 2)          AS total_sales,
  o.Target_Sales,
  ROUND((SUM(s.Sales_Amount) - o.Target_Sales) / o.Target_Sales * 100, 2) AS sales_gap_pct
FROM v_employee_kpi e
JOIN v_outlet_master o  ON e.Outlet_ID  = o.Outlet_ID
JOIN v_daily_sales s    ON e.Outlet_ID  = s.Outlet_ID
GROUP BY e.Outlet_ID, o.Location, o.Store_Type, o.Target_Sales;

In [0]:
%sql
SELECT 'kpi_by_outlet'        AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.kpi_by_outlet
UNION ALL
SELECT 'employee_performance'  AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.employee_performance
UNION ALL
SELECT 'kpi_sales_correlation' AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.kpi_sales_correlation;

### Customer Data 

In [0]:
import pandas as pd

base_path = "/Volumes/data_analysis_retail/source_data/raw/Customer_Data/"
file = "Customer_data.xlsx"

customer_df = pd.read_excel(base_path + file, sheet_name="Customer Data", skiprows=1)

# Keep only real customer rows
customer_df = customer_df[
    customer_df["Customer_ID"].notna() &
    customer_df["Customer_ID"].astype(str).str.startswith("CUST")
]

spark.createDataFrame(customer_df).createOrReplaceTempView("v_customer_data")

print(f"✅ Customers: {len(customer_df)} rows — should be 5,000")

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.customer_segments AS
SELECT
  Age_Group,
  Gender,
  COUNT(Customer_ID)                                        AS customer_count,
  ROUND(AVG(Purchase_Frequency), 2)                        AS avg_purchase_frequency,
  ROUND(AVG(Avg_Basket_Value), 2)                          AS avg_basket_value,
  ROUND(AVG(Purchase_Frequency * Avg_Basket_Value), 2)     AS avg_total_spend,
  ROUND(SUM(Purchase_Frequency * Avg_Basket_Value), 2)     AS total_segment_spend,
  CASE
    WHEN AVG(Purchase_Frequency * Avg_Basket_Value) >= 2000 THEN 'High Value'
    WHEN AVG(Purchase_Frequency * Avg_Basket_Value) >= 1000 THEN 'Mid Value'
    ELSE                                                         'Low Value'
  END AS segment_tier
FROM v_customer_data
GROUP BY Age_Group, Gender;

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.spending_by_age AS
SELECT
  Age_Group,
  COUNT(Customer_ID)                                        AS customer_count,
  ROUND(AVG(Purchase_Frequency), 2)                        AS avg_purchase_frequency,
  ROUND(AVG(Avg_Basket_Value), 2)                          AS avg_basket_value,
  ROUND(AVG(Purchase_Frequency * Avg_Basket_Value), 2)     AS avg_total_spend,
  ROUND(SUM(Purchase_Frequency * Avg_Basket_Value), 2)     AS total_spend,
  ROUND(MIN(Purchase_Frequency * Avg_Basket_Value), 2)     AS min_spend,
  ROUND(MAX(Purchase_Frequency * Avg_Basket_Value), 2)     AS max_spend
FROM v_customer_data
GROUP BY Age_Group
ORDER BY Age_Group;

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.location_purchase_patterns AS
SELECT
  Location,
  COUNT(Customer_ID)                                        AS customer_count,
  ROUND(AVG(Purchase_Frequency), 2)                        AS avg_purchase_frequency,
  ROUND(AVG(Avg_Basket_Value), 2)                          AS avg_basket_value,
  ROUND(AVG(Purchase_Frequency * Avg_Basket_Value), 2)     AS avg_total_spend,
  ROUND(SUM(Purchase_Frequency * Avg_Basket_Value), 2)     AS total_spend,
  ROUND(SUM(Purchase_Frequency * Avg_Basket_Value) /
    SUM(SUM(Purchase_Frequency * Avg_Basket_Value)) OVER () * 100, 2) AS spend_share_pct
FROM v_customer_data
GROUP BY Location;

In [0]:
%sql
SELECT 'customer_segments'         AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.customer_segments
UNION ALL
SELECT 'spending_by_age'           AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.spending_by_age
UNION ALL
SELECT 'location_purchase_patterns' AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.location_purchase_patterns;

###  Expansion Recommendation

In [0]:
import pandas as pd

base_path = "/Volumes/data_analysis_retail/source_data/raw/Daily_Sales/"
file = "Daily_Sales.xlsx"

outlet_df = pd.read_excel(base_path + file, sheet_name="Outlet Master", skiprows=1)
sales_df  = pd.read_excel(base_path + file, sheet_name="Daily Sales",   skiprows=1)
cust_df   = pd.read_excel(base_path + file, sheet_name="Customer Data", skiprows=1)
gap_df    = pd.read_excel(base_path + file, sheet_name="Gap Analysis",  skiprows=1)

# Clean all — keep only real rows
outlet_df = outlet_df[outlet_df["Outlet_ID"].notna()   & outlet_df["Outlet_ID"].astype(str).str.startswith("OT")]
sales_df  = sales_df[sales_df["Outlet_ID"].notna()     & sales_df["Outlet_ID"].astype(str).str.startswith("OT")]
cust_df   = cust_df[cust_df["Customer_ID"].notna()     & cust_df["Customer_ID"].astype(str).str.startswith("CUST")]
gap_df    = gap_df[gap_df["Outlet_ID"].notna()         & gap_df["Outlet_ID"].astype(str).str.startswith("OT")]

spark.createDataFrame(outlet_df).createOrReplaceTempView("v_outlet_master")
spark.createDataFrame(sales_df).createOrReplaceTempView("v_daily_sales")
spark.createDataFrame(cust_df).createOrReplaceTempView("v_customer_data")
spark.createDataFrame(gap_df).createOrReplaceTempView("v_gap_analysis")

print(f"✅ Outlets   : {len(outlet_df)} rows — should be 50")
print(f"✅ Sales     : {len(sales_df)} rows — should be 18,300")
print(f"✅ Customers : {len(cust_df)} rows — should be 5,000")
print(f"✅ Gap       : {len(gap_df)} rows — should be 50")

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.expansion_location_scorecard AS
SELECT
  o.Location,
  COUNT(DISTINCT o.Outlet_ID)                                          AS existing_outlets,
  ROUND(SUM(s.Sales_Amount), 2)                                        AS total_sales,
  ROUND(SUM(s.Sales_Amount) / COUNT(DISTINCT o.Outlet_ID), 2)         AS avg_sales_per_outlet,
  ROUND(AVG(g.`Gap_%`), 2)                                            AS avg_gap_pct,
  CASE
    WHEN AVG(g.`Gap_%`) > 5   THEN 'Overperforming'
    WHEN AVG(g.`Gap_%`) >= 0  THEN 'On Target'
    ELSE                           'Underperforming'
  END                                                                   AS gap_status
FROM v_outlet_master o
JOIN v_daily_sales s   ON o.Outlet_ID = s.Outlet_ID
JOIN v_gap_analysis g  ON o.Outlet_ID = g.Outlet_ID
GROUP BY o.Location;

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.expansion_customer_demand AS
SELECT
  c.Location,
  COUNT(c.Customer_ID)                                                  AS total_customers,
  ROUND(AVG(c.Purchase_Frequency), 2)                                   AS avg_purchase_frequency,
  ROUND(AVG(c.Avg_Basket_Value), 2)                                     AS avg_basket_value,
  ROUND(AVG(c.Purchase_Frequency * c.Avg_Basket_Value), 2)              AS avg_total_spend,
  ROUND(SUM(c.Purchase_Frequency * c.Avg_Basket_Value), 2)              AS total_customer_spend
FROM v_customer_data c
GROUP BY c.Location;

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.expansion_recommendation AS
SELECT
  sc.Location,
  sc.existing_outlets,
  sc.avg_sales_per_outlet,
  sc.avg_gap_pct,
  sc.gap_status,
  cd.total_customers,
  cd.avg_total_spend,
  ROUND(cd.total_customers / sc.existing_outlets, 1)                    AS customers_per_outlet,
  CASE sc.Location
    WHEN 'Sylhet'      THEN 1
    WHEN 'Mymensingh'  THEN 2
    WHEN 'Rajshahi'    THEN 3
    WHEN 'Dhaka'       THEN 4
    WHEN 'Chittagong'  THEN 5
    WHEN 'Khulna'      THEN 6
    WHEN 'Barisal'     THEN 7
  END                                                                    AS expansion_rank,
  CASE sc.Location
    WHEN 'Sylhet'     THEN 'RECOMMENDED'
    WHEN 'Mymensingh' THEN 'RECOMMENDED'
    WHEN 'Rajshahi'   THEN 'RECOMMENDED'
    ELSE                   'Sufficient Coverage'
  END                                                                    AS recommendation,
  CASE sc.Location
    WHEN 'Sylhet'     THEN 'Highest customers/outlet (133) + highest avg spend (1,324) + only 4 outlets'
    WHEN 'Mymensingh' THEN 'Only 2 outlets with +12% gap performance — existing outlets overloaded'
    WHEN 'Rajshahi'   THEN 'Strong demand (104.8 customers/outlet) + solid avg spend (1,211)'
    ELSE 'Adequate outlet coverage relative to demand'
  END                                                                    AS rationale
FROM data_analysis_retail.gold.expansion_location_scorecard sc
JOIN data_analysis_retail.gold.expansion_customer_demand cd ON sc.Location = cd.Location;

In [0]:
%sql
CREATE OR REPLACE TABLE data_analysis_retail.gold.expansion_final_recommendation AS
SELECT
  expansion_rank                          AS rank,
  Location                                AS recommended_location,
  existing_outlets                        AS current_outlets,
  customers_per_outlet                    AS demand_signal,
  avg_total_spend                         AS avg_customer_spend,
  avg_gap_pct                             AS current_gap_pct,
  rationale
FROM data_analysis_retail.gold.expansion_recommendation
WHERE recommendation = 'RECOMMENDED'
ORDER BY expansion_rank;

In [0]:
%sql
SELECT 'expansion_location_scorecard'  AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.expansion_location_scorecard
UNION ALL
SELECT 'expansion_customer_demand'     AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.expansion_customer_demand
UNION ALL
SELECT 'expansion_recommendation'      AS tbl, COUNT(*) AS rows FROM data_analysis_retail.gold.expansion_recommendation;